In [2]:
import json
import os
os.environ["IMAGEIO_FFMPEG_EXE"] = "ffmpeg"   # assumes ffmpeg is in your PATH
import re
import gdown
import requests
import tempfile
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from moviepy import ImageClip, AudioFileClip, ColorClip, TextClip, CompositeVideoClip, concatenate_videoclips
from moviepy.video.fx.FadeIn import FadeIn
from moviepy.video.fx.FadeOut import FadeOut

In [3]:
# Khai báo object Video
@dataclass
class Animation:
    type: str
    startTime_sec: Optional[float] = None
    duration_sec: Optional[float] = None
    start_zoom: Optional[float] = None
    end_zoom: Optional[float] = None
    direction: Optional[str] = None

@dataclass
class Position:
    x: int
    y: int
    anchor: str

@dataclass
class Layer:
    layer_id: str
    type: str
    content: Optional[str] = None
    url: Optional[str] = None
    font: Optional[str] = None
    size: Optional[int] = None
    color: Optional[str] = None
    position: Optional[Position] = None
    animation: Optional[Animation] = None

@dataclass
class Scene:
    scene_id: int
    slide_id: int
    audioUrl: Optional[str]
    audioDuration_sec: float
    layers: List[Layer] = field(default_factory=list)

@dataclass
class VideoMetadata:
    title: str
    resolution: str
    fps: int

@dataclass
class VideoProject:
    metadata: VideoMetadata
    scenes: List[Scene]

In [4]:
load_dotenv()
GOOGLE_FONTS_API_KEY = os.getenv("GOOGLE_FONTS_API_KEY") #API key trong .env 
if GOOGLE_FONTS_API_KEY is None:
    raise ValueError("GOOGLE_FONTS_API_KEY NOT FOUND")

# Hàm đọc JSON
def load_project_from_json(json_path: str) -> VideoProject:
    with open(json_path, "r") as f:
        data = json.load(f)

    metadata = VideoMetadata(
        title=data["videoMetadata"]["title"],
        resolution=data["videoMetadata"]["resolution"],
        fps=data["videoMetadata"]["fps"]
    )

    scenes = []
    for s in data["scenes"]:
        layers = []
        for l in s["layers"]:
            position = Position(**l["position"]) if "position" in l else None
            animation = Animation(**l["animation"]) if "animation" in l else None
            layers.append(Layer(
                layer_id=l["layer_id"],
                type=l["type"],
                content=l.get("content"),
                url=l.get("url"),
                font=l.get("font"),
                size=l.get("size"),
                color=l.get("color"),
                position=position,
                animation=animation
            ))
        scenes.append(Scene(
            scene_id=s["scene_id"],
            slide_id=s["slide_id"],
            audioUrl=s.get("audioUrl"),
            audioDuration_sec=s["audioDuration_sec"],
            layers=layers
        ))

    return VideoProject(metadata=metadata, scenes=scenes)

def extract_file_id(drive_url: str) -> Optional[str]:
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", drive_url)
    return match.group(1) if match else None

def download_to_temp(url: str) -> Optional[str]:
    file_id = extract_file_id(url)
    if not file_id:
        return None
    tmp_dir = tempfile.gettempdir()
    tmp_filename = os.path.join(tmp_dir, file_id)
    gdown.download(f"https://drive.google.com/uc?export=download&id={file_id}", tmp_filename, quiet=True)
    return tmp_filename

def fetch_fallback_font(output_dir: str = "fonts") -> str:
    fallback_url = "https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSans.ttf"
    fallback_path = os.path.join(output_dir, "DejaVuSans.ttf")
    os.makedirs(output_dir, exist_ok=True)
    if not os.path.exists(fallback_path):
        print("Downloading fallback font DejaVuSans...")
        r = requests.get(fallback_url)
        if r.status_code == 200:
            with open(fallback_path, "wb") as f:
                f.write(r.content)
        else:
            raise RuntimeError("Failed to download fallback font DejaVuSans")
    return fallback_path

def fetch_google_font_via_api(font_name: str, api_key: str, output_dir: str = "fonts") -> str:
    os.makedirs(output_dir, exist_ok=True)
    api_url = f"https://www.googleapis.com/webfonts/v1/webfonts?key={api_key}"
    try:
        r = requests.get(api_url)
        if r.status_code != 200:
            print(f"⚠️ Could not fetch font list from Google API. Using fallback.")
            return fetch_fallback_font(output_dir)
        data = r.json()
        family_entry = next((f for f in data["items"] if f["family"].lower() == font_name.lower()), None)
        if not family_entry:
            print(f"⚠️ Font '{font_name}' not found. Using fallback.")
            return fetch_fallback_font(output_dir)
        font_url = family_entry["files"].get("regular")
        if not font_url:
            print(f"⚠️ No regular variant found for {font_name}. Using fallback.")
            return fetch_fallback_font(output_dir)
        font_path = os.path.join(output_dir, f"{font_name.replace(' ', '_')}.ttf")
        if not os.path.exists(font_path):
            print(f"Downloading {font_name} from Google Fonts API...")
            resp = requests.get(font_url)
            if resp.status_code == 200:
                with open(font_path, "wb") as f:
                    f.write(resp.content)
            else:
                print(f"⚠️ Failed to download {font_name}, using fallback.")
                return fetch_fallback_font(output_dir)
        return font_path
    except Exception as e:
        print(f"⚠️ Error fetching font {font_name}: {e}. Using fallback.")
        return fetch_fallback_font(output_dir)

def get_font_path(font_name: Optional[str], api_key: str, output_dir: str = "fonts") -> str:
    if font_name:
        return fetch_google_font_via_api(font_name, api_key, output_dir)
    return fetch_fallback_font(output_dir)

def apply_animation_to_clip(clip, layer: Layer, safe_duration: float, canvas_size: Tuple[int, int]):
    if not layer.animation:
        return clip.with_duration(safe_duration)

    anim = layer.animation
    start = anim.startTime_sec if anim.startTime_sec is not None else 0.0
    effect_duration = float(anim.duration_sec) if anim.duration_sec is not None else 0.0

    # Keep the full duration, just shift start time
    clip = clip.with_start(start).with_duration(safe_duration - start)

    anim_type = (anim.type or "").lower()

    if anim_type == "fadein":
        clip = clip.with_effects([FadeIn(duration=effect_duration)])
    elif anim_type == "fadeout":
        clip = clip.with_effects([FadeOut(duration=effect_duration)])
    elif anim_type == "slideinfromleft":
        canvas_w, canvas_h = canvas_size
        final_x = layer.position.x if layer.position else (canvas_w - clip.w) / 2
        final_y = layer.position.y if layer.position else (canvas_h - clip.h) / 2
        start_x = -clip.w

        def pos_fn(t):
            progress = min(max(t / effect_duration, 0.0), 1.0) if effect_duration > 0 else 1.0
            x = start_x + progress * (final_x - start_x)
            return (x, final_y)

        clip = clip.with_position(pos_fn)

    return clip


def apply_kenburns_to_image(clip, anim: Animation, safe_duration: float):
    start_zoom = anim.start_zoom or 1.0
    end_zoom = anim.end_zoom or 1.1

    def scale_fn(t):
        progress = min(max(t / safe_duration, 0.0), 1.0)
        return start_zoom + (end_zoom - start_zoom) * progress

    return clip.resized(scale_fn)

def build_video_from_project(project: VideoProject, api_key: str):
    width, height = map(int, project.metadata.resolution.split("x"))
    scenes_clips = []
    EPSILON = 0.02

    for scene in project.scenes:
        layer_clips = []
        audio_clip = None

        scene_duration = scene.audioDuration_sec
        if scene.audioUrl:
            audio_path = download_to_temp(scene.audioUrl)
            if audio_path:
                audio_clip = AudioFileClip(audio_path)
                scene_duration = min(scene_duration, audio_clip.duration)

        safe_duration = max(0, scene_duration - EPSILON)

        for layer in scene.layers:
            if layer.type == "image" and layer.url:
                img_path = download_to_temp(layer.url)
                if img_path:
                    img_clip = ImageClip(img_path).resized((width, height)).with_duration(safe_duration)
                    if layer.animation and (layer.animation.type or "").lower() == "kenburns":
                        img_clip = apply_kenburns_to_image(img_clip, layer.animation, safe_duration)
                    img_clip = apply_animation_to_clip(img_clip, layer, safe_duration, (width, height))
                    layer_clips.append(img_clip)

            elif layer.type == "color" and layer.color:
                rgb = tuple(int(layer.color.lstrip('#')[i:i+2], 16) for i in (0, 2, 4))
                color_clip = ColorClip(size=(width, height), color=rgb)
                color_clip = apply_animation_to_clip(color_clip, layer, safe_duration, (width, height))
                layer_clips.append(color_clip)

            elif layer.type == "text" and layer.content:
                font_path = get_font_path(layer.font, api_key)
                txt_clip = TextClip(
                    text=layer.content,
                    font=font_path,
                    font_size=layer.size or 40,
                    color=layer.color or 'white',
                    size=(width, None),
                    method="caption"
                )
                if layer.position:
                    txt_clip = txt_clip.with_position((layer.position.x, layer.position.y))
                txt_clip = apply_animation_to_clip(txt_clip, layer, safe_duration, (width, height))
                layer_clips.append(txt_clip)

        scene_clip = CompositeVideoClip(layer_clips, size=(width, height)).with_duration(safe_duration)
        if audio_clip:
            scene_clip = scene_clip.with_audio(audio_clip.subclipped(0, safe_duration))

        scenes_clips.append(scene_clip)

    final_video = concatenate_videoclips(scenes_clips, method="compose")
    return final_video

# Example usage
project = load_project_from_json("scene_composition_agent_output.json")
video = build_video_from_project(project, GOOGLE_FONTS_API_KEY)
video.write_videofile(
    "GPUfinal_output.mp4",
    codec="h264_nvenc",          # GPU encoder
    fps=project.metadata.fps,
    audio_codec="aac",
    ffmpeg_params=["-preset", "fast"]  # optional: "p1" (fastest) → "p7" (slowest, better quality)
)


frame_index:   1%|          | 271/23743 [05:42<1:35:13,  4.11it/s, now=None]

MoviePy - Building video GPUfinal_output.mp4.
MoviePy - Writing audio in GPUfinal_outputTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video GPUfinal_output.mp4



frame_index:   1%|          | 145/23743 [00:56<3:01:55,  2.16it/s, now=None]

KeyboardInterrupt: 